# test_vision_extraction_on_gt_joined

Wariant `test_vision_extraction_on_gt` — **bez** matchingu 1:1 po typie/indeksie dla tekstu.

Powód: niektóre modele (np. `gpt-4.1-mini`) poprawnie wydobywają treść, ale inaczej ją porcjują (scalają/rozbijają elementy). Stary test dawał wtedy fałszywe „fail".

Tu łączymy wszystkie elementy tekstowe strony w jeden joined text i porównujemy `SequenceMatcher.ratio()` z analogicznie złączonym GT. Weryfikacja tabel — **identyczna** jak w oryginale (markdown exact po normalizacji + opis-sim informacyjnie).

Testy:
1. **Test A — joined text similarity** (tekstowe typy zgrupowane w jeden blok per strona, `ratio ≥ JOINED_SIM_THRESHOLD`).
2. **Test B — section-header count** (informacyjnie, bez fail).
3. **Test C — tabele** (liczba tabel + `md_exact` po normalizacji; jak w oryginalnym notebooku).

## 1. Setup

In [37]:
%load_ext autoreload
%autoreload 2

import re
import difflib
import json
import os
import sys
import warnings
from collections import Counter
from pathlib import Path

import anthropic
import openai
import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.app.config import PDF_PATH, VISION_MODEL
from backend.app.document.image_renderer import apply_cropboxes, page_to_base64
from backend.app.document.structure_extractor import extract_structure
from backend.app.document.vision_prompt import VISION_SYSTEM_PROMPT_v2
from backend.app.document.vision_response import build_section_hint, clean_response

load_dotenv(ROOT / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Brak ANTHROPIC_API_KEY w env/.env"
if not os.getenv("OPENAI_API_KEY"):
    warnings.warn("Brak OPENAI_API_KEY — modele gpt-* zostaną pominięte.")

EVAL_DIR = ROOT / "data" / "eval"
GT_GLOB = "bgk_extraction*.json"

MODELS = [
    "claude-haiku-4-5-20251001",
    "claude-sonnet-4-6",
    "gpt-4.1-mini",
    "gpt-4o",
    "gpt-4.1",
]

# Typy wchodzące w joined text (wszystko prozaiczne — tekst, listy, podpisy itp.)
TEXTUAL_TYPES = {"text", "list", "identifier", "caption", "footnote", "subsection-header"}
# Nagłówki — nie w joined text, liczone osobno jako metryka pomocnicza
HEADER_TYPES = {"section-header"}
# Tabele — osobny test (jak w oryginale)
TABLE_TYPE = "table"
# Ignorowane przy joined text i przy innych metrykach tekstowych
IGNORE_TYPES = {"picture", "infographic"}

JOINED_SIM_THRESHOLD = 0.99

PRICING: dict[str, tuple[float, float]] = {
    "claude-haiku-4-5-20251001": (1.00, 5.00),
    "claude-sonnet-4-6":         (3.00, 15.00),
    "gpt-4.1-nano":              (0.10, 0.40),
    "gpt-4o-mini":               (0.15, 0.60),
    "gpt-4.1-mini":              (0.40, 1.60),
    "gpt-4o":                    (2.50, 10.00),
    "gpt-4.1":                   (2.00, 8.00),
}

print(f"ROOT:   {ROOT}")
print(f"PDF:    {PDF_PATH}")
print(f"EVAL:   {EVAL_DIR}")
print(f"MODELS: {MODELS}")
print(f"default VISION_MODEL: {VISION_MODEL}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
ROOT:   c:\mirror\zadanie\docquery
PDF:    C:\mirror\zadanie\docquery\data\raw\raport_2024_pl.pdf
EVAL:   c:\mirror\zadanie\docquery\data\eval
MODELS: ['claude-haiku-4-5-20251001', 'claude-sonnet-4-6', 'gpt-4.1-mini', 'gpt-4o', 'gpt-4.1']
default VISION_MODEL: claude-haiku-4-5-20251001


## 2. Wczytanie plików GT

In [38]:
gt_files = sorted(EVAL_DIR.glob(GT_GLOB))
assert gt_files, f"Brak plików GT pasujących do {GT_GLOB} w {EVAL_DIR}"

gts = []
for p in gt_files:
    data = json.loads(p.read_text(encoding="utf-8"))
    gts.append({"file": p.name, "page": data["page"], "elements": data["elements"]})

gt_overview = pd.DataFrame(
    [
        {
            "file": g["file"],
            "page": g["page"],
            "n_elements": len(g["elements"]),
            "types": dict(Counter(e["element_type"] for e in g["elements"])),
        }
        for g in gts
    ]
)
display(gt_overview)

,file,page,n_elements,types
0,bgk_extraction_p054_gt.json,54,13,"{'section-header': 1, 'identifier': 1, 'text':..."
1,bgk_extraction_p059_gt.json,59,16,"{'caption': 1, 'table': 1, 'text': 12, 'subsec..."
2,bgk_extraction_p095_gt.json,95,13,"{'picture': 3, 'text': 8, 'subsection-header': 2}"


## 3. Helpers — joined text, tabele, normalizacja, diff

- `join_textual(elements)` — skleja `text` pól elementów z `TEXTUAL_TYPES` w kolejności listy, separator `\n\n`.
- `count_section_headers(elements)` — liczba `section-header`.
- `normalize_md_table` + `_split_table_text` — re-use logiki z `test_vision_extraction_on_gt.ipynb`.
- `render_html_diff` — pomoc przy diagnostyce różnic.

In [39]:
def join_textual(elements: list[dict]) -> str:
    parts = [e["text"] for e in elements if e["element_type"] in TEXTUAL_TYPES]
    return "\n\n".join(parts)


def count_section_headers(elements: list[dict]) -> int:
    return sum(1 for e in elements if e["element_type"] == "section-header")


def _split_table_text(text: str) -> tuple[str, str]:
    parts = text.split("\n\n", 1)
    return (parts[0], parts[1]) if len(parts) == 2 else ("", text)


def normalize_md_table(md: str) -> str:
    """Usuwa kosmetyczne różnice w spacjach/myślnikach w markdown-tabeli."""
    lines = md.strip().splitlines()
    normalized = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if re.match(r'^\|[\s\-:|]+\|[\s\-:|]*$', stripped):
            sep_cells = stripped.split('|')
            new_cells = []
            for cell in sep_cells:
                c = cell.strip()
                if not c:
                    new_cells.append('')
                    continue
                left = ':' if c.startswith(':') else ''
                right = ':' if c.endswith(':') and c != ':' else ''
                new_cells.append(f'{left}-{right}')
            normalized.append('|'.join(new_cells))
        else:
            cells = stripped.split('|')
            normalized.append('|'.join(c.strip() for c in cells))
    return '\n'.join(normalized)


def render_html_diff(gt_text: str, pred_text: str) -> str:
    diff = list(difflib.unified_diff(
        gt_text.splitlines(), pred_text.splitlines(),
        fromfile="gt", tofile="pred", lineterm="",
    ))
    out = []
    for line in diff:
        esc = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        if line.startswith("+") and not line.startswith("+++"):
            out.append(f"<span style='color:#2a7'>{esc}</span>")
        elif line.startswith("-") and not line.startswith("---"):
            out.append(f"<span style='color:#c33'>{esc}</span>")
        elif line.startswith("@@"):
            out.append(f"<span style='color:#88a'>{esc}</span>")
        else:
            out.append(esc)
    return "<br>".join(out)


def _compute_cost(model: str, in_tok: int, out_tok: int) -> float:
    in_price, out_price = PRICING.get(model, (0.0, 0.0))
    return in_tok * in_price / 1_000_000 + out_tok * out_price / 1_000_000


def call_vision(client_a, client_o, doc, extracted_page, model):
    """Zwraca (elements, usage). Analogicznie do call_vision w test_vision_extraction_on_gt."""
    hint = build_section_hint(extracted_page)
    image_b64, media_type = page_to_base64(doc, extracted_page)

    if model.startswith("gpt-"):
        if client_o is None:
            raise RuntimeError("Brak OPENAI_API_KEY — gpt-* niedostępne")
        user_content = [
            {"type": "image_url", "image_url": {"url": f"data:{media_type};base64,{image_b64}"}}
        ]
        if hint:
            user_content.append({"type": "text", "text": hint})
        resp = client_o.chat.completions.create(
            model=model,
            max_tokens=8000,
            messages=[
                {"role": "system", "content": VISION_SYSTEM_PROMPT_v2},
                {"role": "user", "content": user_content},
            ],
            response_format={"type": "json_object"},
        )
        raw = resp.choices[0].message.content
        in_tok = resp.usage.prompt_tokens
        out_tok = resp.usage.completion_tokens
    else:
        content = [
            {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_b64}}
        ]
        if hint:
            content.append({"type": "text", "text": hint})
        resp = client_a.messages.create(
            model=model,
            max_tokens=20000,
            system=VISION_SYSTEM_PROMPT_v2,
            messages=[{"role": "user", "content": content}],
        )
        raw = resp.content[0].text
        in_tok = resp.usage.input_tokens
        out_tok = resp.usage.output_tokens

    parsed = clean_response(raw)
    elements = parsed.get("elements", [])
    usage = {
        "input_tokens": in_tok,
        "output_tokens": out_tok,
        "cost_usd": _compute_cost(model, in_tok, out_tok),
    }
    return elements, usage

## 4. Ekstrakcja (strona × model)

In [40]:
# document, doc = extract_structure(PDF_PATH)
# client_a = anthropic.Anthropic()
# client_o = openai.OpenAI() if os.getenv("OPENAI_API_KEY") else None

# predictions: dict[tuple[int, str], list[dict]] = {}
# usages: dict[tuple[int, str], dict] = {}
# errors: dict[tuple[int, str], str] = {}

# for gt in gts:
#     page_num = gt["page"]
#     extracted_page = document.get_page(page_num)
#     if extracted_page is None:
#         print(f"[skip] nie znaleziono strony {page_num} w strukturze dokumentu")
#         continue
#     apply_cropboxes(doc, [extracted_page])
#     for model in MODELS:
#         key = (page_num, model)
#         try:
#             elements, usage = call_vision(client_a, client_o, doc, extracted_page, model)
#             predictions[key] = elements
#             usages[key] = usage
#             print(
#                 f"[{model} p{page_num}] {len(elements)} elementów, "
#                 f"in={usage['input_tokens']} out={usage['output_tokens']} "
#                 f"cost=${usage['cost_usd']:.5f}"
#             )
#         except Exception as exc:
#             errors[key] = repr(exc)
#             print(f"[{model} p{page_num}] BŁĄD: {exc!r}")

# doc.close()
# print(f"\ngotowych predykcji: {len(predictions)}, błędów: {len(errors)}")

## 5. Test A — joined text similarity (cała strona jako jedna grupa)

Główna metryka. Łączymy wszystkie elementy o typie z `TEXTUAL_TYPES` w jeden string (separator `\n\n`) i porównujemy `SequenceMatcher(None, gt_joined, pred_joined).ratio()`.

Pass ↔ `sim ≥ JOINED_SIM_THRESHOLD` (start: `0.90`).

In [41]:
joined_rows = []
joined_lookup: dict[tuple[int, str], dict] = {}

for gt in gts:
    page_num = gt["page"]
    gt_joined = join_textual(gt["elements"])
    for model in MODELS:
        key = (page_num, model)
        if key not in predictions:
            continue
        pred_joined = join_textual(predictions[key])
        sim = difflib.SequenceMatcher(None, gt_joined, pred_joined).ratio()
        sim = round(sim, 3)
        passed = sim >= JOINED_SIM_THRESHOLD
        joined_rows.append({
            "page": page_num,
            "model": model,
            "gt_chars": len(gt_joined),
            "pred_chars": len(pred_joined),
            "joined_sim": sim,
            "joined_pass": passed,
        })
        joined_lookup[key] = {
            "gt_joined": gt_joined,
            "pred_joined": pred_joined,
            "joined_sim": sim,
            "joined_pass": passed,
        }

joined_df = pd.DataFrame(joined_rows)
display(joined_df)
display(Markdown("**Pivot `joined_sim`:**"))
display(joined_df.pivot(index="model", columns="page", values="joined_sim"))

,page,model,gt_chars,pred_chars,joined_sim,joined_pass
0,54,claude-haiku-4-5-20251001,4411,4329,0.922,False
1,54,claude-sonnet-4-6,4411,4411,1.000,True
2,54,gpt-4.1-mini,4411,4411,0.998,True
3,54,gpt-4o,4411,4411,0.987,False
4,54,gpt-4.1,4411,4419,0.994,True
5,59,claude-haiku-4-5-20251001,4371,4339,0.954,False
6,59,claude-sonnet-4-6,4371,4371,1.000,True
7,59,gpt-4.1-mini,4371,4400,0.996,True
8,59,gpt-4o,4371,4371,0.991,True
9,59,gpt-4.1,4371,4377,0.988,False


**Pivot `joined_sim`:**

page,54,59,95
model,,,
claude-haiku-4-5-20251001,0.922,0.954,0.875
claude-sonnet-4-6,1.000,1.000,0.997
gpt-4.1,0.994,0.988,0.996
gpt-4.1-mini,0.998,0.996,0.999
gpt-4o,0.987,0.991,0.995


## 6. Test B — liczba section-header (metryka pomocnicza)

Informacyjnie: czy model odtwarza tę samą liczbę sekcji co GT. Bez wpływu na pass/fail.

In [42]:
sh_rows = []
sh_lookup: dict[tuple[int, str], dict] = {}

for gt in gts:
    page_num = gt["page"]
    gt_sh = count_section_headers(gt["elements"])
    for model in MODELS:
        key = (page_num, model)
        if key not in predictions:
            continue
        pred_sh = count_section_headers(predictions[key])
        sh_rows.append({
            "page": page_num,
            "model": model,
            "gt_sh_count": gt_sh,
            "pred_sh_count": pred_sh,
            "sh_match": gt_sh == pred_sh,
        })
        sh_lookup[key] = {"gt_sh_count": gt_sh, "pred_sh_count": pred_sh}

sh_df = pd.DataFrame(sh_rows)
display(sh_df)

,page,model,gt_sh_count,pred_sh_count,sh_match
0,54,claude-haiku-4-5-20251001,1,1,True
1,54,claude-sonnet-4-6,1,1,True
2,54,gpt-4.1-mini,1,1,True
3,54,gpt-4o,1,1,True
4,54,gpt-4.1,1,1,True
5,59,claude-haiku-4-5-20251001,0,0,True
6,59,claude-sonnet-4-6,0,0,True
7,59,gpt-4.1-mini,0,0,True
8,59,gpt-4o,0,0,True
9,59,gpt-4.1,0,0,True


## 7. Test C — tabele (identycznie jak w `test_vision_extraction_on_gt`)

Match 1:1 po kolejności. Dla każdej pary (gt_table, pred_table):
- `desc_sim` = `SequenceMatcher.ratio()` opisu (informacyjnie),
- `md_exact` = `normalize_md_table(gt).strip() == normalize_md_table(pred).strip()`,
- `md_sim` = `SequenceMatcher.ratio()` na znormalizowanym markdown.

Fail per (page, model) jeśli liczba tabel różna lub jakakolwiek `md_exact == False`.

In [43]:
table_rows = []
table_lookup: dict[tuple[int, str], dict] = {}
table_mismatches: list[tuple] = []

for gt in gts:
    page_num = gt["page"]
    gt_tables = [e for e in gt["elements"] if e["element_type"] == TABLE_TYPE]
    for model in MODELS:
        key = (page_num, model)
        if key not in predictions:
            continue
        pred_tables = [e for e in predictions[key] if e["element_type"] == TABLE_TYPE]

        if not gt_tables and not pred_tables:
            table_lookup[key] = {
                "n_tables_gt": 0, "n_tables_pred": 0,
                "md_exact_all": None, "md_sim_min": None,
            }
            continue

        md_exacts = []
        md_sims = []
        for i in range(max(len(gt_tables), len(pred_tables))):
            g = gt_tables[i] if i < len(gt_tables) else None
            p = pred_tables[i] if i < len(pred_tables) else None
            if g is None or p is None:
                table_rows.append({
                    "page": page_num, "model": model, "table_idx": i,
                    "gt_present": g is not None, "pred_present": p is not None,
                    "desc_sim": None, "md_exact": False, "md_sim": None,
                })
                md_exacts.append(False)
                continue

            g_desc, g_md = _split_table_text(g["text"])
            p_desc, p_md = _split_table_text(p["text"])
            g_md_norm = normalize_md_table(g_md)
            p_md_norm = normalize_md_table(p_md)

            desc_sim = round(difflib.SequenceMatcher(None, g_desc, p_desc).ratio(), 3) \
                if (g_desc or p_desc) else None
            md_exact = g_md_norm == p_md_norm
            md_sim = round(difflib.SequenceMatcher(None, g_md_norm, p_md_norm).ratio(), 3)

            table_rows.append({
                "page": page_num, "model": model, "table_idx": i,
                "gt_present": True, "pred_present": True,
                "desc_sim": desc_sim, "md_exact": md_exact, "md_sim": md_sim,
            })
            md_exacts.append(md_exact)
            md_sims.append(md_sim)
            if not md_exact:
                table_mismatches.append((page_num, model, i, g_md_norm, p_md_norm))

        table_lookup[key] = {
            "n_tables_gt": len(gt_tables),
            "n_tables_pred": len(pred_tables),
            "md_exact_all": all(md_exacts) if md_exacts else None,
            "md_sim_min": min(md_sims) if md_sims else None,
        }

tables_df = pd.DataFrame(table_rows)
display(tables_df)

,page,model,table_idx,gt_present,pred_present,desc_sim,md_exact,md_sim
0,59,claude-haiku-4-5-20251001,0,True,True,0.385,True,1.000
1,59,claude-sonnet-4-6,0,True,True,0.806,True,1.000
2,59,gpt-4.1-mini,0,True,True,0.000,False,0.289
3,59,gpt-4o,0,True,True,0.326,False,0.851
4,59,gpt-4.1,0,True,True,0.682,True,1.000


## 8. Raport zbiorczy

`joined_sim`, `joined_pass`, counts nagłówków, metryki tabel, koszt.

In [44]:
summary_rows = []
for gt in gts:
    page_num = gt["page"]
    for model in MODELS:
        key = (page_num, model)
        if key not in predictions:
            continue
        j = joined_lookup[key]
        s = sh_lookup[key]
        t = table_lookup[key]
        u = usages.get(key, {})
        summary_rows.append({
            "page": page_num,
            "model": model,
            "joined_sim": j["joined_sim"],
            "joined_pass": j["joined_pass"],
            "gt_sh": s["gt_sh_count"],
            "pred_sh": s["pred_sh_count"],
            "n_tables_gt": t["n_tables_gt"],
            "n_tables_pred": t["n_tables_pred"],
            "tables_md_exact": t["md_exact_all"],
            "worst_md_sim": t["md_sim_min"],
            "cost_usd": round(u.get("cost_usd", 0.0), 5),
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

display(Markdown("**Pivot `joined_sim` (model × page):**"))
display(summary_df.pivot(index="model", columns="page", values="joined_sim"))

display(Markdown("**Pivot `tables_md_exact` (model × page):**"))
display(summary_df.pivot(index="model", columns="page", values="tables_md_exact"))

,page,model,joined_sim,joined_pass,gt_sh,pred_sh,n_tables_gt,n_tables_pred,tables_md_exact,worst_md_sim,cost_usd
0,54,claude-haiku-4-5-20251001,0.922,False,1,1,0,0,None,NaN,0.01230
1,54,claude-sonnet-4-6,1.000,True,1,1,0,0,None,NaN,0.03724
2,54,gpt-4.1-mini,0.998,True,1,1,0,0,None,NaN,0.00436
3,54,gpt-4o,0.987,False,1,1,0,0,None,NaN,0.02035
4,54,gpt-4.1,0.994,True,1,1,0,0,None,NaN,0.01675
5,59,claude-haiku-4-5-20251001,0.954,False,0,0,1,1,True,1.000,0.01408
6,59,claude-sonnet-4-6,1.000,True,0,0,1,1,True,1.000,0.04278
7,59,gpt-4.1-mini,0.996,True,0,0,1,1,False,0.289,0.00476
8,59,gpt-4o,0.991,True,0,0,1,1,False,0.851,0.02541
9,59,gpt-4.1,0.988,False,0,0,1,1,True,1.000,0.01952


**Pivot `joined_sim` (model × page):**

page,54,59,95
model,,,
claude-haiku-4-5-20251001,0.922,0.954,0.875
claude-sonnet-4-6,1.000,1.000,0.997
gpt-4.1,0.994,0.988,0.996
gpt-4.1-mini,0.998,0.996,0.999
gpt-4o,0.987,0.991,0.995


**Pivot `tables_md_exact` (model × page):**

page,54,59,95
model,,,
claude-haiku-4-5-20251001,None,True,None
claude-sonnet-4-6,None,True,None
gpt-4.1,None,True,None
gpt-4.1-mini,None,False,None
gpt-4o,None,False,None


In [48]:
summary_df.groupby("model").agg(cost_usd=("cost_usd", "sum"))


,cost_usd
model,
claude-haiku-4-5-20251001,0.03766
claude-sonnet-4-6,0.11231
gpt-4.1,0.05248
gpt-4.1-mini,0.01289
gpt-4o,0.06374


In [49]:
170/3*0.11232

6.3648

In [50]:
170/3*0.05248

2.9738666666666664

In [52]:
170/3*0.01289

0.7304333333333334

## 9. Diagnostyka — unified diff joined text dla przypadków `joined_sim < threshold`

Renderujemy diff GT vs pred (joined), żeby zobaczyć gdzie model się rozjeżdża — brakujące akapity, dodane interpretacje, itp.

In [45]:
# Wpisz podzbiór MODELS, żeby ograniczyć szum. Pusta lista = pokaż wszystkie poniżej progu.
DIAG_MODELS: list[str] = ['gpt-4.1']

unknown = [m for m in DIAG_MODELS if m not in MODELS]
assert not unknown, f"DIAG_MODELS zawiera modele spoza MODELS: {unknown}"

to_show = [
    (page, model) for (page, model), j in joined_lookup.items()
    if not j["joined_pass"] and (not DIAG_MODELS or model in DIAG_MODELS)
]

if not to_show:
    display(Markdown("_Żadnego przypadku poniżej progu — wszystko pass._"))
else:
    for page, model in sorted(to_show):
        j = joined_lookup[(page, model)]
        display(Markdown(f"### p{page} / `{model}` — joined_sim={j['joined_sim']}"))
        display(HTML(render_html_diff(j["gt_joined"], j["pred_joined"])))

### p59 / `gpt-4.1` — joined_sim=0.988

## 10. Diagnostyka tabel — diff znormalizowanego markdown (dla `md_exact=False`)

In [46]:
if not table_mismatches:
    display(Markdown("_Wszystkie tabele zgodne po normalizacji._"))
else:
    for page, model, idx, g_md, p_md in table_mismatches:
        display(Markdown(f"### p{page} / `{model}` / table #{idx}"))
        display(HTML(render_html_diff(g_md, p_md)))

### p59 / `gpt-4.1-mini` / table #0

### p59 / `gpt-4o` / table #0

# Wnioski 
Najbardziej uniwersalnym modelem który poprawnie wydobywa zarówno tekst jak i tabele jest Sonnet.
gpt 4.1- mini świetnie radzi sobie z tekstem nie zachowuje jednak poprawnej struktóry tabel.
Gpt 4.1 robi błędy językowe jednak zachowuję struktóre table.

Pod względem kosztu Sonnet jest najdroższy - koszt ekstrakcji całego dokumentu to około 6,4$
Gpt-4.1-mini 0,7$ a gpt-4.1 około 3$.

Do table potrzebny jest sonnet albo gpt-4.1.

Dwie opcje: ablo zapłacić za sonnet albo rozbudowywać pipeline extrakcyjny. Dwókrotne odpytanie LLM albo OCR + LLM.